# 1. Imports

In [2]:
import matplotlib.pyplot as plt
import pandas
import numpy
from pandas import DataFrame
from IPython.display import Markdown, display

# 2. Data Loading

In [3]:
dataset: DataFrame = pandas.read_csv("data/caso_full.csv")

## 2.1 Schema Overview

In [4]:
display(Markdown(f"**Amount of tuples**: {len(dataset)}"))

display(Markdown(f"**Column names and its types**:"))

columns_names_with_type: DataFrame = DataFrame(
    {
        "column_name": dataset.columns,
        "type": dataset.dtypes.values
    }    
)

columns_names_with_type

**Amount of tuples**: 3853648

**Column names and its types**:

,column_name,type
0,city,object
1,city_ibge_code,float64
2,date,object
3,epidemiological_week,int64
4,estimated_population,float64
5,estimated_population_2019,float64
6,is_last,bool
7,is_repeated,bool
8,last_available_confirmed,int64
9,last_available_confirmed_per_100k_inhabitants,float64


### Sample

In [5]:
dataset.head(10)

,city,city_ibge_code,date,epidemiological_week,estimated_population,estimated_population_2019,is_last,is_repeated,last_available_confirmed,last_available_confirmed_per_100k_inhabitants,last_available_date,last_available_death_rate,last_available_deaths,order_for_place,place_type,state,new_confirmed,new_deaths
0,Rio Branco,1200401.0,2020-03-17,202012,413418.0,407319.0,False,False,3,0.72566,2020-03-17,0.0,0,1,city,AC,3,0
1,NaN,12.0,2020-03-17,202012,894470.0,881935.0,False,False,3,0.33539,2020-03-17,0.0,0,1,state,AC,3,0
2,Rio Branco,1200401.0,2020-03-18,202012,413418.0,407319.0,False,False,3,0.72566,2020-03-18,0.0,0,2,city,AC,0,0
3,NaN,12.0,2020-03-18,202012,894470.0,881935.0,False,False,3,0.33539,2020-03-18,0.0,0,2,state,AC,0,0
4,Rio Branco,1200401.0,2020-03-19,202012,413418.0,407319.0,False,False,4,0.96754,2020-03-19,0.0,0,3,city,AC,1,0
5,NaN,12.0,2020-03-19,202012,894470.0,881935.0,False,False,4,0.44719,2020-03-19,0.0,0,3,state,AC,1,0
6,Rio Branco,1200401.0,2020-03-20,202012,413418.0,407319.0,False,False,7,1.69320,2020-03-20,0.0,0,4,city,AC,3,0
7,NaN,12.0,2020-03-20,202012,894470.0,881935.0,False,False,7,0.78259,2020-03-20,0.0,0,4,state,AC,3,0
8,Rio Branco,1200401.0,2020-03-21,202012,413418.0,407319.0,False,False,11,2.66075,2020-03-21,0.0,0,5,city,AC,4,0
9,NaN,12.0,2020-03-21,202012,894470.0,881935.0,False,False,11,1.22978,2020-03-21,0.0,0,5,state,AC,4,0


### Memory Usage Before Optimization

In [6]:
dataset_total_memory_usage = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage:.2f} MB"
    )
)

**Total memory usage:** 1553.11 MB

# 3. Memory Optimization

In [7]:
def optimize_dtypes(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    conversions = {
        "city":                      ("category", None),
        "city_ibge_code":            ("Int32",    lambda c: c.round()),
        "date":                      ("datetime", None),
        "last_available_date":       ("datetime", None),
        "estimated_population":      ("Int32",    None),
        "estimated_population_2019": ("Int32",    None),
        "place_type":                ("category", lambda c: c.replace({"city": "C", "state": "S"})),
    }
    rows = []
    for col, (dtype, pre) in conversions.items():
        before = df[col].memory_usage(deep=True) / 1024**2
        if pre:
            df[col] = pre(df[col])
        if dtype == "datetime":
            df[col] = pandas.to_datetime(df[col])
        else:
            df[col] = df[col].astype(dtype)
        after = df[col].memory_usage(deep=True) / 1024**2
        rows.append({
            "column":      col,
            "before_MB":   round(before, 3),
            "after_MB":    round(after, 3),
            "reduction_%": round((before - after) / before * 100, 1),
        })
    return df, DataFrame(rows)

dataset, memory_report = optimize_dtypes(dataset)
memory_report

,column,before_MB,after_MB,reduction_%
0,city,288.845,7.876,97.3
1,city_ibge_code,29.401,18.376,37.5
2,date,246.234,29.401,88.1
3,last_available_date,246.234,29.401,88.1
4,estimated_population,29.401,18.376,37.5
5,estimated_population_2019,29.401,18.376,37.5
6,place_type,224.202,3.675,98.4


## Memory Usage After Optimization

In [8]:
dataset_total_memory_usage_after_optimization = dataset.memory_usage(deep = True).sum() / (1024 ** 2)

display(
    Markdown(
        f"**Total memory usage:** {dataset_total_memory_usage_after_optimization:.2f} MB"
    )
)

**Total memory usage:** 584.87 MB

In [ ]:
before    = dataset_total_memory_usage
after     = dataset_total_memory_usage_after_optimization
saved     = before - after
reduction = (saved / before) * 100

labels = ["Before Optimization", "After Optimization"]
values = [before, after]
colors = ["#e05c5c", "#4caf7d"]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(labels, values, color=colors, width=0.5)

ax.set_title(
    f"Memory reduced by {reduction:.1f}%  ({saved:.0f} MB saved)",
    fontsize=13, pad=14
)
ax.set_ylabel("Memory Usage (MB)")
ax.set_ylim(0, max(values) * 1.2)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(values) * 0.02,
        f"{value:.0f} MB",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.annotate(
    "",
    xy=(1, after), xytext=(0, before),
    xycoords=("data", "data"), textcoords=("data", "data"),
    arrowprops=dict(arrowstyle="-|>", color="gray", lw=1.5)
)

ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# 4. Missing Values Analysis

## Summary

In [10]:
display(
    Markdown(f"**Total amount of missing values**: {dataset.isna().sum().sum()}")
)

**Total amount of missing values**: 90223

## Per-Column Detail

In [11]:
empty_data_reckoning_rendered: DataFrame = DataFrame(dataset.isna().sum().sort_values(ascending = True))
empty_data_reckoning_rendered

,0
last_available_confirmed,0
state,0
place_type,0
order_for_place,0
last_available_deaths,0
last_available_death_rate,0
last_available_date,0
new_confirmed,0
new_deaths,0
is_last,0


### city

In [12]:
missing_city_mask = (dataset["city"].isna()) | (dataset["city"] == "Importados/Indefinidos")

empty_city_values = dataset.loc[
    missing_city_mask,
    ["city", "state", "city_ibge_code"]
]

empty_city_values

,city,state,city_ibge_code
1,NaN,AC,12
3,NaN,AC,12
5,NaN,AC,12
7,NaN,AC,12
9,NaN,AC,12
...,...,...,...
3853087,NaN,TO,17
3853227,NaN,TO,17
3853367,NaN,TO,17
3853507,NaN,TO,17


In [13]:
missing_city_rate: float = len(empty_city_values) / len(dataset)

display(
    Markdown(
        f"**Missing city rate:** {missing_city_rate:.2f}%"
    )
)

display(
    Markdown(
        "**So... drop it!**" if missing_city_rate < 0.25 else "**Consider imputing values**"
    )
)

**Missing city rate:** 0.01%

**So... drop it!**

### last_available_confirmed_per_100k_inhabitants

In [14]:
missing_last_available_per_100k_inhabitants_mask = dataset["last_available_confirmed_per_100k_inhabitants"].isna()

empty_values_last_100k = dataset.loc[
    missing_last_available_per_100k_inhabitants_mask,
    ["last_available_confirmed_per_100k_inhabitants", "city", "state", "city_ibge_code"]
]

empty_values_last_100k

,last_available_confirmed_per_100k_inhabitants,city,state,city_ibge_code
431,NaN,Manoel Urbano,AC,1200344
436,NaN,Porto Walter,AC,1200393
453,NaN,Manoel Urbano,AC,1200344
458,NaN,Porto Walter,AC,1200393
475,NaN,Manoel Urbano,AC,1200344
...,...,...,...,...
3768460,NaN,Mateiros,TO,1712702
3768600,NaN,Mateiros,TO,1712702
3768740,NaN,Mateiros,TO,1712702
3768880,NaN,Mateiros,TO,1712702


### estimated_population_2019

In [15]:
missing_estimated_population_2019_mask = (dataset["estimated_population_2019"].isna() | dataset["estimated_population_2019"] == "<NA>")

estimated_population = dataset.loc[
    missing_estimated_population_2019_mask,
    ["estimated_population", "estimated_population_2019", "city", "state", "city_ibge_code"]
]

estimated_population

,estimated_population,estimated_population_2019,city,state,city_ibge_code


### estimated_population

In [16]:
missing_estimated_population_mask = (dataset["estimated_population"].isna())

missing_estimated_population_lines = dataset.loc[
    missing_estimated_population_mask,
    ["estimated_population", "city", "state", "city_ibge_code"]
]

missing_estimated_population_lines

,estimated_population,city,state,city_ibge_code
16196,<NA>,Importados/Indefinidos,AL,<NA>
16200,<NA>,Importados/Indefinidos,AL,<NA>
16204,<NA>,Importados/Indefinidos,AL,<NA>
16208,<NA>,Importados/Indefinidos,AL,<NA>
16212,<NA>,Importados/Indefinidos,AL,<NA>
...,...,...,...,...
3754873,<NA>,Importados/Indefinidos,SP,<NA>
3755520,<NA>,Importados/Indefinidos,SP,<NA>
3756167,<NA>,Importados/Indefinidos,SP,<NA>
3756814,<NA>,Importados/Indefinidos,SP,<NA>


### city_ibge_code

In [17]:
missing_city_ibge_mask = (dataset["city_ibge_code"].isna()) | (dataset["city_ibge_code"] == "<NA>")

missing_city_ibge_list = dataset.loc[
    missing_city_ibge_mask,
    ["city", "state", "city_ibge_code"]
]

missing_city_ibge_list

,city,state,city_ibge_code
16196,Importados/Indefinidos,AL,<NA>
16200,Importados/Indefinidos,AL,<NA>
16204,Importados/Indefinidos,AL,<NA>
16208,Importados/Indefinidos,AL,<NA>
16212,Importados/Indefinidos,AL,<NA>
...,...,...,...
3754873,Importados/Indefinidos,SP,<NA>
3755520,Importados/Indefinidos,SP,<NA>
3756167,Importados/Indefinidos,SP,<NA>
3756814,Importados/Indefinidos,SP,<NA>


In [18]:
dataset.loc[
    dataset["last_available_confirmed_per_100k_inhabitants"].isna(),
    "estimated_population"
].describe()

count         15520.0
mean     10645.107023
std      80295.381959
min             776.0
25%            3308.0
50%            5341.0
75%           10560.0
max         4039277.0
Name: estimated_population, dtype: Float64

In [19]:
DataFrame(dataset.isna().sum().sort_values(ascending=False))

,0
last_available_confirmed_per_100k_inhabitants,29166
city,20119
estimated_population,13646
estimated_population_2019,13646
city_ibge_code,13646
date,0
epidemiological_week,0
is_last,0
is_repeated,0
last_available_confirmed,0


## 4.4 Drop Decision

In [20]:
missing_columns = [
    "city",
    "last_available_confirmed_per_100k_inhabitants",
    "estimated_population",
    "estimated_population_2019",
    "city_ibge_code"
]

missing_summary = DataFrame({
    "Missing Count": dataset[missing_columns].isna().sum(),
    "Missing Rate":  dataset[missing_columns].isna().mean()
}).sort_values("Missing Rate", ascending=False)

display(
    missing_summary
    .style
    .format({"Missing Rate": "{:.2%}"})
    .background_gradient(cmap="Reds", subset=["Missing Rate"])
    .set_caption("Missing Values Summary")
)

,Missing Count,Missing Rate
last_available_confirmed_per_100k_inhabitants,29166,0.76%
city,20119,0.52%
estimated_population,13646,0.35%
estimated_population_2019,13646,0.35%
city_ibge_code,13646,0.35%


In [21]:
rows_missing = dataset[missing_columns].isna().any(axis=1)

summary_stats = DataFrame({
    "Metric": ["Rows with missing values", "Rate of affected rows"],
    "Value": [
        f"{rows_missing.sum():,}",
        f"{rows_missing.mean():.2%}"
    ]
})

display(
    summary_stats
    .style
    .set_caption("Dataset Impact Summary")
)

,Metric,Value
0,Rows with missing values,"49,279"
1,Rate of affected rows,1.28%


**Decision:** Rows with missing values account for only 1.28% of the dataset and are concentrated in `city`, `estimated_population`, `city_ibge_code`, and `last_available_confirmed_per_100k_inhabitants`. Since all affected rows correspond to unidentifiable entries or state-level records already excluded by the `place_type` filter, they are safely dropped without meaningful data loss.

# 5. Data Integrity

## 5.1 Repeated Records (is_repeated)

In [22]:
repeated_counts = dataset["is_repeated"].value_counts()

display(Markdown(f"**Repeated records:** {repeated_counts.get(True, 0):,}"))
display(Markdown(f"**Non-repeated records:** {repeated_counts.get(False, 0):,}"))

dataset[dataset["is_repeated"]].head(5)

**Repeated records:** 1,015,645

**Non-repeated records:** 2,838,003

,city,city_ibge_code,date,epidemiological_week,estimated_population,estimated_population_2019,is_last,is_repeated,last_available_confirmed,last_available_confirmed_per_100k_inhabitants,last_available_date,last_available_death_rate,last_available_deaths,order_for_place,place_type,state,new_confirmed,new_deaths
579,Acrelândia,1200013,2020-05-20,202021,15490,15256,False,True,74,477.72757,2020-05-19,0.0135,1,53,C,AC,0,0
580,Assis Brasil,1200054,2020-05-20,202021,7534,7417,False,True,2,26.54632,2020-05-19,0.5000,1,20,C,AC,0,0
581,Brasiléia,1200104,2020-05-20,202021,26702,26278,False,True,6,22.47023,2020-05-19,0.0000,0,12,C,AC,0,0
582,Bujari,1200138,2020-05-20,202021,10420,10266,False,True,18,172.74472,2020-05-19,0.0000,0,43,C,AC,0,0
583,Capixaba,1200179,2020-05-20,202021,12008,11733,False,True,10,83.27781,2020-05-19,0.0000,0,13,C,AC,0,0


**Decision:** `is_repeated` days were interpolated rather than zeroed out because this flag indicates that the state health secretariat did not publish a bulletin that day — not that no cases occurred. Keeping zero values would bias the LSTM into learning spurious dips in the series.

## 5.2 Latest Record Flag (is_last)

In [23]:
is_last_counts = dataset["is_last"].value_counts()
total = len(dataset)

display(Markdown(f"**is_last = True:** {is_last_counts.get(True, 0):,} ({is_last_counts.get(True, 0) / total * 100:.1f}%)"))
display(Markdown(f"**is_last = False:** {is_last_counts.get(False, 0):,} ({is_last_counts.get(False, 0) / total * 100:.1f}%)"))

dataset[dataset["is_last"]].head(5)

**is_last = True:** 5,616 (0.1%)

**is_last = False:** 3,848,032 (99.9%)

,city,city_ibge_code,date,epidemiological_week,estimated_population,estimated_population_2019,is_last,is_repeated,last_available_confirmed,last_available_confirmed_per_100k_inhabitants,last_available_date,last_available_death_rate,last_available_deaths,order_for_place,place_type,state,new_confirmed,new_deaths
12861,Acrelândia,1200013,2021-11-05,202144,15490,15256,True,False,1796,11594.57715,2021-11-05,0.0206,37,587,C,AC,0,0
12862,Assis Brasil,1200054,2021-11-05,202144,7534,7417,True,False,1827,24250.06637,2021-11-05,0.0131,24,554,C,AC,0,0
12863,Brasiléia,1200104,2021-11-05,202144,26702,26278,True,False,3000,11235.11347,2021-11-05,0.0147,44,546,C,AC,1,0
12864,Bujari,1200138,2021-11-05,202144,10420,10266,True,False,1144,10978.88676,2021-11-05,0.0149,17,577,C,AC,0,0
12865,Capixaba,1200179,2021-11-05,202144,12008,11733,True,False,675,5621.25250,2021-11-05,0.0252,17,547,C,AC,0,0


**Decision:** `is_last` was filtered to `True` to retain only the most recent, corrected version of each record, avoiding double-counting of cases that were later revised.

## 5.3 Negative Values in new_confirmed

In [24]:
negatives = dataset[dataset["new_confirmed"] < 0]

display(Markdown(f"**Negative new_confirmed records:** {len(negatives):,}"))

if len(negatives) > 0:
    display(negatives[["date", "state", "city", "new_confirmed"]].head(10))

**Negative new_confirmed records:** 38,638

,date,state,city,new_confirmed
252,2020-05-02,AC,Cruzeiro do Sul,-3
280,2020-05-04,AC,Mâncio Lima,-1
303,2020-05-06,AC,Acrelândia,-1
328,2020-05-07,AC,Senador Guiomard,-1
354,2020-05-09,AC,Cruzeiro do Sul,-1
460,2020-05-14,AC,Rodrigues Alves,-1
485,2020-05-15,AC,Senador Guiomard,-4
491,2020-05-16,AC,Brasiléia,-1
504,2020-05-16,AC,Rodrigues Alves,-1
514,2020-05-17,AC,Bujari,-1


**Decision:** `new_confirmed` was clipped at zero because negative values do not indicate a decrease in cases; they reflect reclassifications between municipalities by the secretariat. Clipping preserves series integrity without introducing artificial drops.

# 6. Data Coverage

In [25]:
state_level = dataset[
    (dataset["place_type"] == "S") &
    (dataset["is_last"] == True)
]

coverage = (
    state_level.groupby("state")["date"]
    .agg(start="min", end="max", days="count")
    .sort_values("days")
)

display(Markdown(f"**Date range:** {state_level['date'].min().date()} → {state_level['date'].max().date()}"))
display(Markdown(f"**States present:** {state_level['state'].nunique()} / 27"))
coverage

**Date range:** 2022-03-25 → 2022-03-27

**States present:** 27 / 27

,start,end,days
state,,,
AC,2022-03-27,2022-03-27,1
SE,2022-03-27,2022-03-27,1
SC,2022-03-27,2022-03-27,1
RS,2022-03-27,2022-03-27,1
RR,2022-03-26,2022-03-26,1
RO,2022-03-27,2022-03-27,1
RN,2022-03-26,2022-03-26,1
RJ,2022-03-27,2022-03-27,1
PR,2022-03-27,2022-03-27,1


# 7. Data Cleaning Pipeline

`place_type` was filtered to state level because state-aggregated series are 
less noisy than city-level data and provide denser, more consistent coverage.

`is_repeated` days were interpolated rather than zeroed out because this flag 
indicates that the state health secretariat did not publish a bulletin that day 
— not that no cases occurred. Keeping zero values would bias the LSTM into 
learning spurious dips in the series.

`new_confirmed` was clipped at zero because negative values do not indicate a 
decrease in cases; they reflect reclassifications between municipalities by the 
secretariat. Clipping preserves series integrity without introducing artificial 
drops.

`is_last` was filtered to True to retain only the most recent, corrected version 
of each record, avoiding double-counting of cases that were later revised.


In [ ]:
df = dataset[
    (dataset["place_type"] == "S") &
    (dataset["is_last"] == True)
].copy().sort_values(["state", "date"])

def clean_state_series(group):
    group = group.copy()
    group["new_confirmed"] = (
        group["new_confirmed"]
        .where(~group["is_repeated"])
        .interpolate(method="linear")
        .clip(lower=0)
    )
    return group

df = pandas.concat([
    clean_state_series(group)
    for _, group in df.groupby("state")
])

display(Markdown(f"**Records after cleaning:** {len(df):,}"))
display(Markdown(f"**Date range:** {df['date'].min().date()} → {df['date'].max().date()}"))
display(Markdown(f"**States:** {df['state'].nunique()}"))
df.head()